In [1]:
from modpath import modify_path

modify_path()

In [ ]:
import pathlib
from src import config, model, pipelines

In [3]:
FIGS_DIR = pathlib.Path.cwd().parent.joinpath('figs/01/')

# Model v1.0

Create the base model to see how it performs.

Removed: `gender`

In [ ]:
model_version = config.ModelVersion.V1_0
data, test = pipelines.load_data(
    train_table=model_version.get_table_name(config.DatasetSplit.TRAIN),
    test_table=model_version.get_table_name(config.DatasetSplit.TEST),
)

In [5]:
drop_cols = ['gender']
untransformed_features = ['is_loyal_customer', 'is_personal_travel']
numeric_features, categorical_features = pipelines.get_model_features(
    drop_cols=drop_cols, untransformed_features=untransformed_features
)

In [6]:
exp_result_v1_0 = pipelines.run_model(
    data,
    test,
    config.TARGET,
    model.objective,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    untransformed_features=untransformed_features,
)

  0%|          | 0/10 [00:00<?, ?it/s]

In [7]:
output_path_v1_0 = FIGS_DIR.joinpath(f'{model_version.version_str}/')
output_path_v1_0.mkdir(parents=True, exist_ok=True)
pipelines.save_figures(exp_result_v1_0, path=output_path_v1_0)
pipelines.score_model(exp_result_v1_0)

 98%|===================| 4876/5000 [00:27<00:00]        

,accuracy,precision,recall,f1-score,AUC,support
Test,0.959,0.960,0.959,0.959,0.994,25976
Train,0.961,0.961,0.961,0.961,0.994,103904
Test (shuffle),0.555,0.471,0.555,0.413,0.407,25976
Train (shuffle),0.584,0.696,0.584,0.455,0.652,103904


<p align="center">
    <img src="../figs/01/v1.0/test_eval.png" width="750px" alt="test_eval_v1.0">
    <br>
    <em>Model v1.0 evaluation on the test set</em>
</p>

<div align="center">

| <center>Dataset</center> | Accuracy | Precision | Recall | F1-Score | ROC AUC |
| :--- | ---: | ---: | ---: | ---: | ---: |
| Test | 0.959 | 0.960 | 0.959 | 0.959 | 0.994 |
| Train | 0.961 | 0.961 | 0.961 | 0.961 | 0.994 |
| Test (Shuffled) | 0.555 | 0.471 | 0.555 | 0.413 | 0.407 |
| Train (Shuffled) | 0.584 | 0.696 | 0.584 | 0.455 | 0.652 |

</div>

The initial model performed suspiciously well with an AUC of 0.994 for both the training and test sets. We tried shuffling the target variable and retraining to see if the model is learning the noise in the data.

The target-shuffled model achieved an AUC of around 0.65 on the training set, meaning it was overfitting the training data. About 97% of its predictions were non-satisfied, but there were still 2200 passengers for which is was able to correctly memorize that they were satisfied. To bring down model complexity, we next tried restricting the range of the model hyperparameters.

# Model v1.1

Restrict the range of the hyperparameters during the optuna study.

In [ ]:
model_version = config.ModelVersion.V1_1
data, test = pipelines.load_data(
    train_table=model_version.get_table_name(config.DatasetSplit.TRAIN),
    test_table=model_version.get_table_name(config.DatasetSplit.TEST),
)

In [9]:
exp_result_v1_1 = pipelines.run_model(
    data,
    test,
    config.TARGET,
    model.objective_restricted,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    untransformed_features=untransformed_features,
)

  0%|          | 0/10 [00:00<?, ?it/s]

In [10]:
output_path_v1_1 = FIGS_DIR.joinpath(f'{model_version.version_str}/')
output_path_v1_1.mkdir(parents=True, exist_ok=True)
pipelines.save_figures(exp_result_v1_1, path=output_path_v1_1)
pipelines.save_shap_plots(
    exp_result_v1_1, output_path_v1_1, 'shap_top_5', max_display=5
)
pipelines.score_model(exp_result_v1_1)

,accuracy,precision,recall,f1-score,AUC,support
Test,0.948,0.948,0.948,0.948,0.990,25976
Train,0.947,0.947,0.947,0.947,0.990,103904
Test (shuffle),0.559,0.484,0.559,0.407,0.388,25976
Train (shuffle),0.569,0.616,0.569,0.420,0.560,103904


<div align="center">

| <center>Dataset</center> | Accuracy | Precision | Recall | F1-Score | ROC AUC |
| :--- | ---: | ---: | ---: | ---: | ---: |
| Test | 0.948 | 0.948 | 0.948 | 0.948 | 0.990 |
| Train | 0.947 | 0.947 | 0.947 | 0.947 | 0.990 |
| Test (Shuffled) | 0.559 | 0.484 | 0.559 | 0.407 | 0.388 |
| Train (Shuffled) | 0.569 | 0.616 | 0.569 | 0.420 | 0.560 |

</div>

The AUC for the shuffled training data dropped to 0.56, which was an improvement. It still predicted almost exclusively non-satisfied for each passenger, though the number of correct satisfied predictions dropped to only around 500.

<p align="center">
    <img src="../figs/01/v1.1/shap_bar.png" width="750px" alt="bar_v1.1">
    <br>
    <em>Mean absolute SHAP values for all features of model v1.1.</em>
    <br></br>
    <img src="../figs/01/v1.1/shap_top_5_beeswarm.png" width="750px" alt="beeswarm_top_5_v1.1">
    <br>
    <em>Beeswarm SHAP values for the top 5 features of model v1.1.</em>
</p>

We checked the SHAP values for each of the features and made the following observations:

1. Customers who rated the wifi service as a 0 appeared on both ends of the SHAP spectrum. This suggested that 0 is not really 0 and is instead "N/A".

2. Economy and Economy Plus contributed almost nothing to the model. What was really important was whether or not the passenger was in business class. For the next model, this feature was changed simply to business or non-business.

3. Arrival delay had some impact on the model while departure delay had essentially none. Since these two were very highly correlated, we dropped departure delay in the next model.

4. A number of the bottom features contributed very little to the model overall and/or lacked a clear color gradient in the SHAP beeswarm plot, so these were dropped for the next model iteration.

We investigated passengers who rated wifi service with a 0 and found that they tended to be flying in business class for business reasons and were loyal customers. These passengers might be frequent flyers who don't rely on wifi to occupy themselves. This may be a legitimate passenger demographic, but given that a 0 rating is (likely) "N/A", it could correspond to many different reasons. We went through the various rating features and replaced all the 0s with a more neutral 3 to prevent the model from learning about passengers based on effectively a non-response (`food_drink`, `flight_distance`, `gate_location`, `online_booking_ease`, `age`).

## Passengers with 0 rated wifi service

In [11]:
pipelines.save_wifi_zero_breakdown_plot(data, output_path_v1_1)

<p align="center">
    <img src="../figs/01/v1.1/wifi_zero_breakdown.png" width="750px" alt="wifi_zero">
    <em>Feature distribution for passengers that rated wifi service a 0.</em>
</p>

Passengers who rated wifi service a 0 were almost all satisfied, tended to be flying for business reasons in business class, and were loyal customers. These passengers might be frequent flyers who don't rely on wifi to occupy themselves. This may be a legitimate passenger demographic, but given that a 0 rating is (likely) "N/A", it could correspond to many different reasons. We decided to let the model infer this type of passenger based on the features and replaced all 0s with 3s to try to prevent overfitting.